# HATS read path over S3
Reads the same four columns of the same HATS partitions with several readers and reduces each to one mean, so the
readers can be compared on wall time and bytes fetched. It is the S3 counterpart of a POSIX read-path script: the
catalog is read from an S3 endpoint instead of a directory.

This notebook is self-contained and does not import the rest of the repository. The harness that records the same
measurements as result files is `ibench-storage hats` (see `storage/scenarios/genesis/hats-reads/README.md`).

**Needs** `uv sync --extra hats`, and these environment variables (or a key file, see the credentials cell):
`IBENCH_S3_ENDPOINT`, `IBENCH_S3_BUCKET`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`. A read-only key is enough.

In [ ]:
import logging
import os
import stat
import threading
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import dask
import lsdb
import nested_pandas as npd
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.fs as pafs
import pyarrow.parquet as pq
import s3fs
from upath import UPath

logging.disable(logging.WARNING)
warnings.filterwarnings("ignore")

## Settings
`CATALOG` is a HATS catalog directory (not a collection) relative to the bucket. `N_PARTITIONS` counts the first
partitions present, in pixel order; 0 reads all of them. `THREADS` is the number of reader threads in this process,
and the number of Dask threads for lsdb. `VARIANTS` picks the readers:

| Variant | Reads with |
|---|---|
| `npd_direct` | `nested_pandas.read_parquet` on an S3 path |
| `lsdb` | `open_catalog`, `map_partitions`, `compute` |
| `pa_arrow_native` | `pyarrow.parquet.read_table` through pyarrow's own C++ S3 client |
| `pa_arrow_ranges` | `read_table` through fsspec, fetching only the column ranges it needs |
| `pa_arrow` | `read_table` through s3fs with its default read-ahead cache (slow over a WAN, see below) |

In [ ]:
ENDPOINT = os.environ.get("IBENCH_S3_ENDPOINT", "https://sdfdatas3.slac.stanford.edu")
BUCKET = os.environ.get("IBENCH_S3_BUCKET", "rubin-benchmarks")
CATALOG = os.environ.get("IBENCH_HATS_CATALOG", "lsdb/gaia_dr3/gaia")
COLUMN = "phot_g_mean_mag"
N_PARTITIONS = int(os.environ.get("IBENCH_HATS_PARTITIONS", 16))
THREADS = int(os.environ.get("IBENCH_HATS_THREADS", 8))
REPS = 2
VARIANTS = ["npd_direct", "lsdb", "pa_arrow_native", "pa_arrow_ranges"]
ENV_FILE = "~/.config/ibench/ro.env"

## Credentials
Taken from the environment. If they are not set, they are read from `ENV_FILE` (lines of `NAME=value`), which must
have mode 600. Only the names are printed.

In [ ]:
if not (os.environ.get("AWS_ACCESS_KEY_ID") and os.environ.get("AWS_SECRET_ACCESS_KEY")):
    path = Path(ENV_FILE).expanduser()
    assert stat.S_IMODE(path.stat().st_mode) == 0o600, f"chmod 600 {path}"
    for line in path.read_text().splitlines():
        line = line.strip().removeprefix("export ")
        if line and not line.startswith("#") and "=" in line:
            k, _, v = line.partition("=")
            os.environ[k.strip()] = v.strip().strip("'\"")
assert os.environ.get("AWS_ACCESS_KEY_ID") and os.environ.get("AWS_SECRET_ACCESS_KEY"), "S3 credentials missing"
print([k for k in os.environ if k.startswith("AWS_")])

## Reaching S3
Three clients are built from the same endpoint and key:

- **s3fs** (`fs`) for everything that goes through fsspec: the Parquet footer reads, `pa_arrow_ranges`, and the
  `pa_arrow` read-ahead reader. The gateway needs path-style addressing, and `max_pool_connections` must cover the
  number of reader threads, otherwise threads wait for a connection.
- **`UPath`** (`upath`) carries the same s3fs options inside a path object. `nested_pandas.read_parquet` and
  `lsdb.open_catalog` accept it, which is how those two are pointed at S3 without any other change.
- **pyarrow's native S3 filesystem** (`nfs`), a separate C++ client that does not go through Python I/O.

In [ ]:
POOL = max(64, THREADS * 2)
FS_OPTS = dict(
    key=os.environ["AWS_ACCESS_KEY_ID"],
    secret=os.environ["AWS_SECRET_ACCESS_KEY"],
    endpoint_url=ENDPOINT,
    client_kwargs={"region_name": "us-east-1"},
    config_kwargs={"s3": {"addressing_style": "path"}, "max_pool_connections": POOL},
)
fs = s3fs.S3FileSystem(**FS_OPTS)
scheme, host = ENDPOINT.split("://")
nfs = pafs.S3FileSystem(
    access_key=FS_OPTS["key"], secret_key=FS_OPTS["secret"], endpoint_override=host, scheme=scheme, region="us-east-1"
)


def upath(key: str) -> UPath:
    return UPath(f"s3://{key}", **FS_OPTS)

### Counting bytes on the wire
The two s3fs entry points that fetch data are wrapped to add up the payload bytes. Dividing by the bytes the four
columns occupy in the files (below) gives the read amplification of a reader. The native client is not counted.

In [ ]:
class Wire:
    n = 0
    lock = threading.Lock()


def _count(n: int) -> None:
    with Wire.lock:
        Wire.n += n


if not getattr(s3fs.core.S3File._fetch_range, "_counted", False):
    _fetch, _cat = s3fs.core.S3File._fetch_range, s3fs.S3FileSystem._cat_file

    def counted_fetch(self, start, end):
        data = _fetch(self, start, end)
        _count(len(data))
        return data

    async def counted_cat(self, *args, **kwargs):
        data = await _cat(self, *args, **kwargs)
        _count(len(data))
        return data

    counted_fetch._counted = True
    s3fs.core.S3File._fetch_range = counted_fetch
    s3fs.S3FileSystem._cat_file = counted_cat

## Partitions
The partition list comes from the catalog metadata (`get_healpix_pixels`). The same four columns are read by every
reader: the column to average plus `ra`, `dec` and `_healpix_29`, which lsdb always loads. `NEED` is the compressed
size of those columns, taken from the Parquet footers, so it is what any reader has to fetch at minimum. `HEAD
latency` is one S3 request per partition and shows the per-request cost before any data moves.

In [ ]:
COLS = [COLUMN, "ra", "dec", "_healpix_29"]
ROOT = f"{BUCKET}/{CATALOG}"
pixels = lsdb.open_catalog(upath(ROOT)).get_healpix_pixels()
if N_PARTITIONS > 0:
    pixels = pixels[:N_PARTITIONS]
PIXELS = [(p.order, p.pixel) for p in pixels]
KEYS = [f"{ROOT}/dataset/Norder={o}/Dir={(p // 10000) * 10000}/Npix={p}.parquet" for o, p in PIXELS]


def head(key):
    t0 = time.perf_counter()
    size = fs.info(key)["size"]
    return size, time.perf_counter() - t0


def need(key):
    md = pq.ParquetFile(fs.open(key, "rb")).metadata
    return sum(
        md.row_group(i).column(j).total_compressed_size
        for i in range(md.num_row_groups)
        for j in range(md.num_columns)
        if md.row_group(i).column(j).path_in_schema in COLS
    )


with ThreadPoolExecutor(THREADS) as ex:
    heads = list(ex.map(head, KEYS))
    NEED = sum(ex.map(need, KEYS))
lat = sorted(t for _, t in heads)
print(f"{len(KEYS)} partitions, {sum(s for s, _ in heads) / 1e9:.2f} GB in the files, {NEED / 1e6:.0f} MB in the 4 columns")
print(f"HEAD latency p50 {lat[len(lat) // 2] * 1e3:.1f} ms, max {lat[-1] * 1e3:.1f} ms")

## Readers
Each reader returns the mean of the column over all the partitions, as a sum and a count per partition so the
partial results combine exactly. The thread pool is the same for every reader except lsdb, which uses Dask's
threaded scheduler with the same number of workers.

`pa_arrow_native` is the fastest here because the whole read, including the decode, runs in C++ without holding the
Python GIL. The s3fs-based readers share one event loop for their I/O.

In [ ]:
def mean_of(parts):
    total, count = 0.0, 0
    for s, c in parts:
        total += s
        count += c
    return total / count


def pool_map(fn):
    with ThreadPoolExecutor(THREADS) as ex:
        return mean_of(ex.map(fn, KEYS))


def arrow_stats(table):
    c = table.column(COLUMN)
    return (pc.sum(c).as_py() or 0.0), pc.count(c, mode="only_valid").as_py()


def pa_arrow():
    return pool_map(lambda k: arrow_stats(pq.read_table(k, columns=COLS, use_threads=False, filesystem=fs)))


def pa_arrow_native():
    return pool_map(lambda k: arrow_stats(pq.read_table(k, columns=COLS, use_threads=False, filesystem=nfs)))


def pa_arrow_ranges():
    from fsspec.parquet import open_parquet_file

    def one(k):
        with open_parquet_file(k, columns=COLS, engine="pyarrow", fs=fs) as f:
            return arrow_stats(pq.read_table(f, columns=COLS, use_threads=False))

    return pool_map(one)


def npd_direct():
    def one(k):
        c = npd.read_parquet(upath(k), columns=COLS)[COLUMN]
        return float(c.sum()), int(c.count())

    return pool_map(one)


def lsdb_compute():
    cat = lsdb.open_catalog(upath(ROOT), columns=[COLUMN], search_filter=lsdb.PixelSearch(PIXELS))

    def part_mean(df, target_column=""):
        c = df[target_column]
        return pd.DataFrame([{"s": float(c.sum()), "n": int(c.count())}])

    meta = pd.DataFrame({"s": pd.Series(dtype="float64"), "n": pd.Series(dtype="int64")})
    with dask.config.set({"num_workers": THREADS}):
        res = cat.map_partitions(part_mean, target_column=COLUMN, meta=meta).compute(progress_bar=False)
    return float(res["s"].sum()) / int(res["n"].sum())


READERS = {
    "pa_arrow": pa_arrow,
    "npd_direct": npd_direct,
    "lsdb": lsdb_compute,
    "pa_arrow_native": pa_arrow_native,
    "pa_arrow_ranges": pa_arrow_ranges,
}

## Run
One untimed warm pass per reader (connections, gateway and filesystem caches), then `REPS` timed passes. `gbps` is the
useful rate: bytes of the four columns over wall time. `over_read` is wire bytes over those bytes; 1.0 means the
reader fetched nothing it did not need. All readers must agree on the mean.

In [ ]:
pa.set_cpu_count(THREADS)
for name in VARIANTS:
    READERS[name]()
rows = []
for rep in range(1, REPS + 1):
    for name in VARIANTS:
        Wire.n = 0
        t0 = time.perf_counter()
        mean = READERS[name]()
        wall = time.perf_counter() - t0
        rows.append(
            {
                "rep": rep,
                "variant": name,
                "partitions": len(KEYS),
                "wall_s": round(wall, 3),
                "gbps": round(NEED * 8 / wall / 1e9, 3),
                "over_read": None if name == "pa_arrow_native" else round(Wire.n / NEED, 2),
                "mean": round(mean, 6),
            }
        )
df = pd.DataFrame(rows)
assert df["mean"].nunique() == 1, df["mean"].unique()
df.groupby("variant")[["wall_s", "gbps", "over_read"]].median().loc[VARIANTS]

Add `"pa_arrow"` to `VARIANTS` to see the default s3fs read-ahead cache: `read_table` on an S3 path fetches large
blocks around every column chunk, so `over_read` is far above 1 and the reader is slow when each request has a
network round trip. `pa_arrow_ranges`, `npd_direct` and `lsdb` fetch the planned column ranges instead.

## Ceiling of this client
Parallel 64 MiB ranged reads of whole partitions with no Parquet decoding. It shows how much one Python process can
pull from the endpoint, and where adding threads stops helping.

In [ ]:
RANGE = 64 << 20


def ceiling(threads):
    keys = [KEYS[i % len(KEYS)] for i in range(threads * 2)]
    t0 = time.perf_counter()
    with ThreadPoolExecutor(threads) as ex:
        total = sum(ex.map(lambda k: len(fs.cat_file(k, start=0, end=RANGE)), keys))
    wall = time.perf_counter() - t0
    return {"threads": threads, "GB": round(total / 1e9, 2), "s": round(wall, 2), "Gbps": round(total * 8 / wall / 1e9, 2)}


pd.DataFrame([ceiling(n) for n in (1, 4, 8, 16, 32)])